# 🧩 Query Decomposition

Some questions are really several questions wearing a trench coat. *"What is LangSmith, and why do
we need it?"* asks for a **definition** and a **justification** — two different things, likely
living in different parts of your corpus.

A single embedding has to average both intents into one vector, and lands somewhere in between —
close to neither. **Decomposition** fixes this by splitting the question into independent
sub-questions, running full RAG on each, then synthesizing one answer from the parts.

```
                  ┌─ "What is LangSmith?"        → retrieve → answer ─┐
"What is          │                                                   │
 LangSmith, and   ├─ "What problems does it solve?" → retrieve → answer ─┼─→ synthesize
 why do we        │                                                   │
 need it?"        └─ "How does it help teams?"    → retrieve → answer ─┘
```

## Learning Objectives
1. **The compound-question problem** — why one vector cannot serve two intents
2. **Reliable decomposition** — using `with_structured_output()` instead of parsing prose
3. **Per-sub-question RAG** — independent retrieval and answering for each part
4. **Synthesis** — merging the Q&A pairs into a single coherent answer
5. **The cost** — decomposition multiplies your LLM and retrieval calls

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Internet access — the corpus is loaded from a live blog URL
- Familiarity with basic RAG (see `01_Introduction_to_RAG/`)

---
## 🧠 Part 1: Why One Query Is Not Enough

Dense retrieval turns your question into a single vector and returns the nearest chunks. That works
when the question has **one** intent.

A compound question breaks the assumption. *"What is LangSmith, and why do we need it?"* contains:

| Sub-intent | Where the answer lives |
|---|---|
| *What is it?* | Definitional passages — overview, product description |
| *Why do we need it?* | Motivational passages — problems, pain points, use cases |

Embedding both together produces a vector that is a **compromise** — it sits between the two
regions and may retrieve chunks that fully satisfy neither. Worse, the top-`k` cutoff means one
sub-intent can be crowded out entirely.

### Key Concepts:
- **Compound question**: one query carrying multiple distinct information needs.
- **Sub-question**: an independently answerable piece of the original.
- **Decomposition**: LLM-driven splitting of the query before retrieval.
- **Synthesis**: recombining the sub-answers into one response.

> **Key Insight**: decomposition trades *cost* for *coverage*. You issue N retrievals instead of
> one, and each is precise, so no sub-intent gets averaged away.

---
## ⚙️ Part 2: Environment Setup

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langsmith import Client

# LangChain core
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Integrations
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

> **Note**: use `LANGSMITH_PROJECT`, not the legacy `LANGCHAIN_PROJECT`. The SDK resolves the
> `LANGSMITH_` prefix **first**, so a `LANGSMITH_PROJECT` already set in `.env` would silently win
> and these traces would land in that project instead of this one.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "Decomposition"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Models

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: LLM (decompose + answer) + embeddings (retrieval)
# ============================================================================
llm = get_experientiallabs_llm()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"🤖 LLM:        {llm.model_name}")
print(f"🔢 Embeddings: {embeddings.model}")

---
## 📚 Part 3: Build the Knowledge Base

The corpus is a single LangChain blog post announcing LangSmith — deliberately small, so retrieval
differences are easy to inspect.

### 3.1 Load and Split

Chunks here are small (100 tokens). Small chunks make the decomposition effect more visible: each
chunk carries roughly one idea, so a compound query genuinely cannot capture several at once.

In [ ]:
# ============================================================================
# KNOWLEDGE BASE: Load the source document and split it into chunks
# ============================================================================
loader = WebBaseLoader(web_paths=("https://blog.langchain.dev/announcing-langsmith",))
blog_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100,    # tokens per chunk — small, so each holds ~one idea
    chunk_overlap=20,  # carry-over to avoid severing sentences
)
splits = text_splitter.split_documents(blog_docs)

print(f"📄 Loaded {len(blog_docs)} document(s) → split into {len(splits)} chunks")

### 3.2 Index Into Chroma

In [ ]:
# ============================================================================
# VECTOR STORE: Embed and index the chunks
# ============================================================================
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print(f"✅ Indexed {len(splits)} chunks into Chroma")

---
## 🔍 Part 4: Baseline — One Query for a Two-Part Question

Establish the comparison first. Retrieve once with the whole compound question and inspect what
comes back, keeping the two sub-intents in mind: *what is it* and *why do we need it*.

In [ ]:
# ============================================================================
# BASELINE: Single retrieval on the full compound question
# ============================================================================
question = "What is LangSmith, and why do we need it?"

baseline_docs = retriever.invoke(question)

print(f"🔍 Single-query retrieval for: {question}")
print(f"📊 {len(baseline_docs)} chunks retrieved\n")
for i, doc in enumerate(baseline_docs, 1):
    print(f"[{i}] {doc.page_content[:150].strip()}...\n")

---
## ✂️ Part 5: Decompose the Question

### 5.1 Why Structured Output Instead of `split("\n")`

An earlier version of this notebook asked for prose and parsed it by splitting on newlines:

```python
generate_queries_decomposition = (
    prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n"))
)
```

That failed in two compounding ways.

**1. The prompt asked for two different things.** It said *"break the input into three
sub-questions"* and then *"generate multiple search queries"*. The model obeyed both and returned
nested markdown — sub-questions **plus** a list of search queries, under headers. That is
compliance, not misbehavior. The original tutorial got away with it because weaker models ignored
the second instruction.

**2. `split("\n")` trusts the model's formatting.** Splitting that markdown yields ~18 list items:
headers like `'### Sub-questions'`, blank strings, and bullet lines. The loop in the next section
then calls `retriever.invoke()` **and** an LLM on every one of them — roughly 18 retrievals and 18
LLM calls instead of 3, most on meaningless input like `''`. Those garbage answers then flow into
the final synthesis step.

Worse, it is *non-deterministic*: the same chain sometimes returned 3 clean lines and sometimes 18,
so the bug appeared and disappeared between runs.

#### The fix

`with_structured_output()` binds a Pydantic schema to the model, so the provider returns validated
JSON that LangChain parses into a `SubQuestions` object. You get a guaranteed `list[str]` — no
headers, no blanks, no prompt-wording roulette.

| | String parsing | Structured output |
|---|---|---|
| **Output type** | Whatever the model emitted | Guaranteed `list[str]` |
| **Formatting drift** | Breaks silently | Impossible — schema-validated |
| **Prompt burden** | Must specify exact layout | Only needs to describe the *task* |
| **Failure mode** | Wrong results, extra cost | Raises immediately |

> **Rule of thumb**: any time you are about to `.split()` or regex an LLM response to get a list or
> a record, reach for `with_structured_output()` instead.

In [ ]:
# ============================================================================
# DECOMPOSITION CHAIN: compound question -> exactly three sub-questions
# ============================================================================
class SubQuestions(BaseModel):
    """Three independent sub-questions that together cover the original question."""

    questions: list[str] = Field(
        description="Exactly three sub-questions, each answerable on its own"
    )


# The prompt asks for ONE thing. Formatting is the schema's job, so there is no
# need to beg the model for "one per line, no markdown, no headers".
template = """Break the input question into exactly three sub-questions.

Each sub-question must be answerable independently, without needing the answers to
the others, and together they should fully cover the original question.

Original question: {question}"""

prompt_decomposition = ChatPromptTemplate.from_template(template)

generate_queries_decomposition = (
    prompt_decomposition
    | llm.with_structured_output(SubQuestions)
    | (lambda x: x.questions)
)

print("✅ Decomposition chain ready")

### 5.2 Run the Decomposition

Note the guaranteed shape: a clean `list[str]` of length 3, with no headers, blanks or bullets to
strip.

In [ ]:
# ============================================================================
# DECOMPOSE: Split the compound question
# ============================================================================
sub_questions = generate_queries_decomposition.invoke({"question": question})

print(f"❓ Original: {question}\n")
print(f"📊 Decomposed into {len(sub_questions)} sub-questions "
      f"(type: {type(sub_questions).__name__} of "
      f"{type(sub_questions[0]).__name__}):\n")
for i, sub_q in enumerate(sub_questions, 1):
    print(f"   {i}. {sub_q}")

---
## 🔗 Part 6: Answer Each Sub-Question Independently

Each sub-question now gets its **own** retrieval and its **own** generation. This is where the
coverage gain comes from — three precise queries instead of one averaged compromise.

`rlm/rag-prompt` is the standard community RAG prompt (context + question → concise answer).

> **Note on `dangerously_pull_public_prompt=True`**: LangSmith refuses to pull public prompts by
> default, since a prompt manifest can carry serialized objects that execute on deserialization.
> The flag is an explicit acknowledgement. The deprecated `hub.pull()` helper cannot pull public
> prompts at all — it never forwards this flag — so the LangSmith SDK is used directly.

In [ ]:
# ============================================================================
# RAG PROMPT: Pull the standard community RAG prompt
# ============================================================================
prompt_rag = Client().pull_prompt(
    "rlm/rag-prompt",
    dangerously_pull_public_prompt=True,   # see the note above
)

print("✅ Pulled 'rlm/rag-prompt'")
print(f"📋 Expects variables: {prompt_rag.input_variables}")

In [ ]:
# ============================================================================
# PER-SUB-QUESTION RAG: retrieve + answer each one separately
# ============================================================================
rag_chain = prompt_rag | llm | StrOutputParser()

rag_results = []
retrieved_per_question = []

for i, sub_question in enumerate(sub_questions, 1):
    retrieved_docs = retriever.invoke(sub_question)
    answer = rag_chain.invoke({"context": retrieved_docs, "question": sub_question})

    rag_results.append(answer)
    retrieved_per_question.append(retrieved_docs)

    print(f"❓ [{i}] {sub_question}")
    print(f"   📄 retrieved {len(retrieved_docs)} chunks")
    print(f"   💬 {answer[:220].strip()}...\n")

### 6.1 Did Decomposition Actually Retrieve More?

The claim is that three targeted queries surface material a single averaged query misses. That is
measurable — compare the set of unique chunks reached by each approach.

In [ ]:
# ============================================================================
# COVERAGE CHECK: unique chunks reached, single query vs decomposition
# ============================================================================
baseline_ids = {doc.page_content for doc in baseline_docs}
decomposed_ids = {doc.page_content for docs in retrieved_per_question for doc in docs}

new_chunks = decomposed_ids - baseline_ids
missed_chunks = baseline_ids - decomposed_ids

print(f"📊 Baseline (1 query)        : {len(baseline_ids)} unique chunks")
print(f"📊 Decomposition (3 queries) : {len(decomposed_ids)} unique chunks")
print(f"✅ Reached ONLY by decomposition : {len(new_chunks)}")
print(f"⚠️  Reached ONLY by the baseline  : {len(missed_chunks)}")

if new_chunks:
    print("\n📄 Example of a chunk the single query missed:")
    print(f"   {sorted(new_chunks)[0][:220].strip()}...")

---
## 🧵 Part 7: Synthesize the Final Answer

The sub-answers are formatted into a numbered Q&A block, which becomes the context for one final
generation. The synthesis prompt never sees the retrieved chunks — only the **answers** — which
keeps the final context small regardless of how many sub-questions were asked.

In [ ]:
# ============================================================================
# FORMATTING: Merge sub-questions and their answers into one context block
# ============================================================================
def format_qa(questions, answers):
    """Render Q&A pairs as numbered text for the synthesis prompt."""
    formatted = ""
    for i, (q, a) in enumerate(zip(questions, answers), start=1):
        formatted += f"Question {i}: {q}\nAnswer {i}: {a}\n\n"
    return formatted.strip()


context = format_qa(sub_questions, rag_results)

print(f"✅ Built synthesis context ({len(context)} characters)\n")
print(context[:600], "...")

In [ ]:
# ============================================================================
# SYNTHESIS: One final answer from the sub-answers
# ============================================================================
synthesis_template = """Here is a set of Q and A:

{context}

Use these to synthesize an answer to the question: {question}
"""

synthesis_prompt = ChatPromptTemplate.from_template(synthesis_template)

final_rag_chain = synthesis_prompt | llm | StrOutputParser()

final_answer = final_rag_chain.invoke({"context": context, "question": question})

print(f"❓ {question}\n")
print(final_answer)

---
## ⚖️ Part 8: The Tradeoff

Decomposition is the most expensive query-transformation technique in this folder. Count the calls
for three sub-questions:

| Step | LLM calls | Retrievals |
|---|---|---|
| Decompose | 1 | 0 |
| Answer each sub-question | 3 | 3 |
| Synthesize | 1 | 0 |
| **Total** | **5** | **3** |

Plain RAG costs 1 and 1. That is a **5× LLM increase** for one user question.

| | Plain RAG | Decomposition |
|---|---|---|
| **Compound questions** | Averages intents, may miss one | Strong — each intent retrieved separately |
| **Simple questions** | Ideal | Wasteful; may fabricate three near-identical sub-questions |
| **Latency** | One round trip | Sequential by default — noticeably slower |
| **Error propagation** | Single point of failure | A bad sub-question quietly poisons the synthesis |

**Use decomposition when** questions genuinely have multiple parts — comparisons, "X and Y"
questions, multi-hop reasoning.

**Skip it when** queries are single-intent, latency-sensitive, or high-volume. Consider routing:
detect compound questions first and decompose only those.

> **Optimization**: the sub-question loop is independent, so it parallelizes. Replacing the `for`
> loop with `rag_chain.batch(...)` cuts wall-clock time substantially at the same token cost.

---
## 📝 Summary

### 1. The Problem
- A compound question has multiple intents, but dense retrieval gives it **one** vector. The result
  is a compromise embedding that can under-serve every sub-intent.

### 2. The Technique
- **Decompose → answer each → synthesize.** Three targeted retrievals replace one averaged query,
  and Part 6.1 measured the extra chunks that surfaced as a result.

### 3. Structured Output Is Not Optional Here
- Parsing prose with `split("\n")` produced ~18 junk "sub-questions" — headers, blanks, bullets —
  each triggering a retrieval and an LLM call.
- The bug was **intermittent**, appearing and vanishing between runs, which is the worst kind.
- `with_structured_output()` with a Pydantic schema guarantees a `list[str]`. Reach for it any time
  you would otherwise `.split()` or regex an LLM response.

### 4. Practical Notes
- Public prompts require `dangerously_pull_public_prompt=True`; the deprecated `hub.pull()` cannot
  fetch them at all, so use the LangSmith `Client`.
- `LANGSMITH_PROJECT` beats the legacy `LANGCHAIN_PROJECT` — set the wrong one and your traces
  silently land in whatever project `.env` names.

### 5. Cost
- 5 LLM calls and 3 retrievals per question, versus 1 and 1 for plain RAG. Justified for genuinely
  multi-part questions; wasteful otherwise.
- The sub-question loop is embarrassingly parallel — use `.batch()` to reclaim the latency.

### Next Steps
- Inspect these runs in LangSmith under the **Decomposition** project; each trace shows the
  decomposition call, three RAG calls, and the synthesis.
- Compare with the sibling techniques: `1. Rewriting or Query Expansion` rephrases *the same*
  question many ways, while decomposition splits it into genuinely *different* questions.